In [1]:
import torch
from transformers import BertTokenizer, BertForSequenceClassification
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from tqdm import tqdm
import pandas as pd
import re

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# read the dataset
df_train = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/data/albert_sft_assignment/train.csv', encoding="Windows-1252")
df_train.rename(columns={
    "sentiment": "polarity",
}, inplace=True)
df_train = df_train[["text", "polarity"]]

# read the dataset
df_test = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/data/albert_sft_assignment/test.csv', encoding="Windows-1252")
df_test.rename(columns={
    "sentiment": "polarity",
}, inplace=True)
df_test = df_test[["text", "polarity"]]

# convert target variables into pytorch tensors
label_map = {"negative": 0, "neutral":1, "positive": 2}

df_train["polarity"] = df_train["polarity"].map(label_map)
df_test["polarity"] = df_test["polarity"].map(label_map)

df_test.head()

,text,polarity
0,Last session of the day http://twitpic.com/67ezh,1.0
1,Shanghai is also really exciting (precisely -...,2.0
2,"Recession hit Veronique Branquinho, she has to...",0.0
3,happy bday!,2.0
4,http://twitpic.com/4w75p - I like it!!,2.0


In [4]:
before = len(df_train)
df_train.dropna(how="any", inplace=True)
print(f"{before - len(df_train)} NaN rows dropped.")

1 NaN rows dropped.


In [5]:
before = len(df_test)
df_test.dropna(how="any", inplace=True)
print(f"{before - len(df_test)} NaN rows dropped.")
df_test['polarity'] = df_test['polarity'].astype('int')

1281 NaN rows dropped.


In [6]:

def text_preprocess(text):
    # Check if the text is a string
    if not isinstance(text, str):
        raise ValueError("Should be a str")

    # Remove urls
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)

    return text

In [7]:
#apply the text_preprocess to
df_train["text_preprocessed"] = df_train["text"].apply(text_preprocess)
df_test["text_preprocessed"] = df_test["text"].apply(text_preprocess)

In [8]:
before = len(df_train)
df_train.drop_duplicates(subset=["text", "polarity"], inplace=True)
print(f"{before - len(df_train)} rows are dropped")
print(f"dataset has {len(df_train)} rows")

0 rows are dropped
dataset has 27480 rows


In [9]:
before = len(df_test)
df_test.drop_duplicates(subset=["text", "polarity"], inplace=True)
print(f"{before - len(df_test)} rows are dropped")
print(f"dataset has {len(df_test)} rows")

0 rows are dropped
dataset has 3534 rows


In [10]:
def is_garbled(text, threshold=0.5):
    non_alpha_ratio = len(re.findall(r'[^a-zA-Z\s]', text)) / max(len(text), 1)
    return non_alpha_ratio > threshold

In [11]:
before = len(df_train)
df_train = df_train[~df_train["text"].apply(is_garbled)]
print(f"{before - len(df_train)} rows are dropped")
print(f"dataset has {len(df_train)} rows")

18 rows are dropped
dataset has 27462 rows


In [12]:
before = len(df_test)
df_test = df_test[~df_test["text"].apply(is_garbled)]
print(f"{before - len(df_test)} rows are dropped")
print(f"dataset has {len(df_test)} rows")

1 rows are dropped
dataset has 3533 rows


In [13]:
model_name = 'bert-base-uncased'
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertForSequenceClassification.from_pretrained(model_name, num_labels=3, ignore_mismatched_sizes=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [14]:
import torch.nn as nn

model.classifier = nn.Sequential(
            nn.Linear(768, 128),  # Input size from ALBERT pooler
            nn.ReLU(),
            nn.Linear(128, 3),
        )
model.init_weights()

In [15]:
model

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [17]:
class CustomDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts.iloc[idx])
        label = self.labels.iloc[idx]

        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )

        return {
            'text': text,
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'label': torch.tensor(label, dtype=torch.long)
        }


In [18]:
# Set maximum sequence length
MAX_LEN = 64

In [19]:
# Create DataLoaders for train and val sets
train_dataset = CustomDataset(df_train['text_preprocessed'], df_train['polarity'], tokenizer, MAX_LEN)
test_dataset = CustomDataset(df_test['text_preprocessed'], df_test['polarity'], tokenizer, MAX_LEN)

In [20]:
# Define training parameters
batch_size = 32
epochs = 30
lr = 2e-5
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

In [21]:
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [22]:
# Training loop
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

In [23]:
device

device(type='cuda')

In [24]:
# Freeze all layers except the classification layer
for param in model.parameters():
    param.requires_grad = False

# Unfreeze the classification layer
for param in model.classifier.parameters():
    param.requires_grad = True

In [26]:
best_val_loss = float('inf')  # Initialize best_val_loss to a very high value
best_epoch = -1  # Initialize best_epoch to an invalid value to track the epoch of the best validation loss

for epoch in range(epochs):
    model.train()
    total_train_loss = 0
    total_val_loss = 0

    # Training
    train_loop = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{epochs} [Training]", leave=True)

    for batch in train_loop:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()

        avg_train_loss = total_train_loss / (train_loop.n + 1)

        # Update the postfix to show the current average training loss
        train_loop.set_postfix(train_loss=avg_train_loss)

    # avg_train_loss = total_train_loss / len(train_loader)

    # Validation
    model.eval()
    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)

            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            total_val_loss += loss.item()

    avg_val_loss = total_val_loss / len(test_loader)

    # Check if the current validation loss is the lowest; if so, save the model
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        best_epoch = epoch
        torch.save(model.state_dict(), 'best_model.pth')  # Save the best model

    print(f"Epoch {epoch + 1}/{epochs}, Training Loss: {avg_train_loss:.4f}, Validation Loss: {avg_val_loss:.4f}")


Epoch 1/30 [Training]: 100%|██████████| 859/859 [01:47<00:00,  8.02it/s, train_loss=1.08]


Epoch 1/30, Training Loss: 1.0832, Validation Loss: 1.0794


Epoch 2/30 [Training]:  17%|█▋        | 149/859 [00:17<01:25,  8.35it/s, train_loss=1.08]


KeyboardInterrupt: 

In [ ]:
def calculate_accuracy(loader, model, device=None):
    """
    Computes overall accuracy for a 3-class classification model.
    Expects model to return logits of shape (batch, 3) and labels to be dtype torch.long with values 0,1,2.
    Returns accuracy as percentage.
    """
    if device is None:
        device = next(model.parameters()).device
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            labels = batch['label'].to(device).long()
            attention_mask = batch['attention_mask'].to(device) # Also need attention mask

            outputs = model(input_ids, attention_mask=attention_mask)                 # logits shape (batch, 3)
            preds = outputs.logits.argmax(dim=1)           # predicted class indices (batch,)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return 100.0 * correct / total if total > 0 else 0.0

In [ ]:
# Print the best epoch and its validation loss
print(f"The lowest validation loss was {best_val_loss:.4f} at epoch {best_epoch + 1}")

# Load the best model and calculate accuracy
model.load_state_dict(torch.load('best_model.pth'))
train_accuracy = calculate_accuracy(train_loader, model, device)
print(f'Best Model Training Accuracy: {train_accuracy:.2f}%')

test_accuracy = calculate_accuracy(test_loader, model, device)
print(f'Best Model Test Accuracy: {test_accuracy:.2f}%')